#**Ticketly Backend Project Setup**

#**Introduction**:
This project is a high-performance Backend API System designed for a ticketing platform like Ticketly.pk. The main goal is to automate ticket bookings, store data securely in a database, and provide a way to retrieve that data for business reporting. It uses modern tools like FastAPI for speed and SQLAlchemy for database management.

# **Environment Setup**
**Logic**: Before starting, we need to install the essential libraries:

**FastAPI**: The framework used to build the API.

**Uvicorn**: The server that runs our FastAPI application.

**SQLAlchemy**: A tool that allows Python to talk to databases without writing complex SQL queries.

**Requests**: Used for testing our own API to see if it works.

In [ ]:
#We are installing the libraries that are required for FastAPI and the database
!pip install fastapi uvicorn sqlalchemy requests

# Summary:
  1. FastAPI: Used to create APIs.
  2. SQLAlchemy: Used to interact with the database.
  3. Requests: Used to test our own API.

**Output Summary**:  "Successfully installed

#**Database Setup with SQLite and SQLAlchemy**

# Database Schema & Models
**Logic & Definition**: In this step, we define the "Skeleton" of our project using an ORM (Object Relational Mapper). Instead of creating tables manually in SQL, we use Python classes.

**Base**: This is the starting point for all our tables.

**Ticket Class**: This defines our table structure with columns: id (Unique ID), movie_name, price, and seat_number.

**create_all**: This command physically creates the .db file in our system.

In [ ]:
# create_engine: used to create a connection to the database
# Column, Integer, String: used to define table columns and their data types
from sqlalchemy import create_engine, Column, Integer, String

# declarative_base: used to create a base class for database models (tables)
from sqlalchemy.ext.declarative import declarative_base

# sessionmaker: used to create database sessions (to run queries like insert, select, update)
from sqlalchemy.orm import sessionmaker


In [ ]:
# Database file name is 'ticketly.db'
# SQLite database will be created in the current folder
DATABASE_URL = "sqlite:///./ticketly.db"

# Create a database engine (connection)
# check_same_thread=False allows the database to be used with FastAPI (multiple requests)
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

# Create a session factory
# autocommit=False → changes are saved only when we commit manually
# autoflush=False → data is not sent to DB automatically
# bind=engine → session is connected to our database engine
SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

# Base class for all database models (tables)
# Every table class will inherit from this Base
Base = declarative_base()


/tmp/ipython-input-1365335032.py:24: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [ ]:
# Table ka structure (id, movie_name, price, seat_number)
class Ticket(Base):
    __tablename__ = "tickets"
    id = Column(Integer, primary_key=True, index=True)
    movie_name = Column(String)
    price = Column(Integer)
    seat_number = Column(String)


In [ ]:
# To Table create
Base.metadata.create_all(bind=engine)
print("Database 'ticketly.db' ban chuka hai aur Table tayyar hai!")


Database 'ticketly.db' ban chuka hai aur Table tayyar hai!


In [ ]:
from sqlalchemy import inspect

# Create an inspector object to examine the database
inspector = inspect(engine)

# Get the names of all tables in the database
tables = inspector.get_table_names()

# Print the table names
print("Tables in database:", tables)


Tables in database: ['tickets']


#summary
We told that our database will have a table named "tickets", in which the movie name and its price will be stored.

#FastAPI Backend Logic

**Logic & Definition**: APIs are like "Waiters" in a restaurant. They take your order (data) to the kitchen (database) and bring back a response.

**POST** (/book-ticket): This endpoint is used to Submit data. It takes movie details from the user and saves them permanently into the database using db.add() and db.commit().

**GET (/view-tickets):** This endpoint is used to Fetch data. It reads everything from the database and shows it in a clean JSON format.

In [ ]:
# FastAPI library to create APIs
# Depends is used to inject dependencies like database session into route functions
from fastapi import FastAPI, Depends

# Session from SQLAlchemy ORM to interact with the database
from sqlalchemy.orm import Session

# Pydantic BaseModel to define the structure of request and response data
from pydantic import BaseModel

# Create a FastAPI app instance
app = FastAPI()

In [ ]:
# Pydantic BaseModel is used for input validation
# It ensures the data sent to the API has the correct structure and types
class TicketCreate(BaseModel):
    # movie_name must be a string
    movie_name: str

    # price must be an integer
    price: int

    # seat_number must be a string
    seat_number: str


In [ ]:
# Function to connect to the database and provide a session
def get_db():
    db = SessionLocal()       # Create a new database session
    try:
        yield db             # Give this session to the API route
    finally:
        db.close()           # Close the session after the request is done

# API route to book a ticket
# Accepts data in the format of TicketCreate
@app.post("/book-ticket")
def book(ticket: TicketCreate, db: Session = Depends(get_db)):
    # Create a new Ticket object using the data from the request
    new_tkt = Ticket(
        movie_name=ticket.movie_name,
        price=ticket.price,
        seat_number=ticket.seat_number
    )

    db.add(new_tkt)          # Add the ticket to the database session
    db.commit()              # Save the ticket to the database
    db.refresh(new_tkt)      # Refresh the object to get updated info (like ID)

    # Return a response confirming the booking
    return {"message": "Ticket Booked!", "data": new_tkt}

# API route to view all tickets
@app.get("/view-tickets")
def view(db: Session = Depends(get_db)):
    # Query the Ticket table and return all tickets
    return db.query(Ticket).all()

# Print a confirmation message in the console
print(" APIs (Book and View) are now defined!")


 APIs (Book and View) are now defined!


# **Start the Server**

# Logic:
 Since we are using Google Colab, we cannot run the server normally because it would block the notebook.

Multiprocessing: We run the server in a separate "Thread" or background process. This allows the API to stay "Live" while we continue to work on other cells.

In [ ]:
# multiprocessing library is used to run multiple processes at the same time
import multiprocessing

# uvicorn is the server used to run FastAPI applications
import uvicorn

# time library is used for handling time-related tasks like delays or measuring time
import time


In [ ]:
# Function to start the FastAPI server using uvicorn
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)  # Run the app locally on port 8000

# Check if this file is being run directly
if __name__ == "__main__":
    # Create a separate process to run the server
    p = multiprocessing.Process(target=start_server)

    # Start the server process
    p.start()

    # Wait 2 seconds to give the server time to start
    time.sleep(2)

    # Print a message confirming the server is live
    print(" Server is live at 127.0.0.1:8000!")


INFO:     Started server process [38738]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


 Server is live at 127.0.0.1:8000!


#**Real World Testing (The Output)**

**Logic**: Data saved in a database file is not easily readable by humans.

**Pandas Integration**: We use the Pandas library to convert the SQL data into a DataFrame (a neat table).

**Purpose**: This makes it easy for managers to see how many tickets were sold in a professional table view.

In [ ]:
# requests library is used to send HTTP requests to the API
import requests

#  Prepare the ticket booking data
data = {
    "movie_name": "The Legend of Maula Jatt",  # Movie name
    "price": 1000,                             # Ticket price
    "seat_number": "B-5"                       # Seat number
}

#  Send a POST request to the FastAPI book-ticket endpoint
res = requests.post("http://127.0.0.1:8000/book-ticket", json=data)

#  Print a heading for clarity
print("--- POST Response (Booking) ---")

#  Print the JSON response returned by the API
print(res.json())


--- POST Response (Booking) ---
{'message': 'Ticket Booked!', 'data': {'price': 1000, 'seat_number': 'B-5', 'id': 3, 'movie_name': 'The Legend of Maula Jatt'}}


In [ ]:
#  Send a GET request to view all tickets
view_res = requests.get("http://127.0.0.1:8000/view-tickets")

# Print a heading for clarity
print("\n--- GET Response (All Tickets) ---")

# Print the JSON response returned by the API (all booked tickets)
print(view_res.json())


--- GET Response (All Tickets) ---
[{'price': 1000, 'seat_number': 'B-5', 'id': 1, 'movie_name': 'The Legend of Maula Jatt'}, {'price': 1000, 'seat_number': 'B-5', 'id': 2, 'movie_name': 'The Legend of Maula Jatt'}, {'price': 1000, 'seat_number': 'B-5', 'id': 3, 'movie_name': 'The Legend of Maula Jatt'}]


**Summary**:
The database has 3 tickets booked.

Each ticket has the following details:

id	movie_name	price	seat_number
1	The Legend of Maula Jatt	1000	B-5
2	The Legend of Maula Jatt	1000	B-5
3	The Legend of Maula Jatt	1000	B-5

Observation:

All tickets are for the same movie.

The price and seat number are identical.

Each ticket has a unique id (1, 2, 3) generated automatically by the database.

# Outcome Summary:

**Automated Workflow**: Successfully created a system where a ticket can be booked via a simple API call.

**Data Persistence**: Even if the server stops, the data remains safe in ticketly.db.

**Report Generation**: The system automatically creates a ticketly_report.csv file, which is ready for Excel.

#Extracting Data from the Database with Pandas

In [ ]:
# pandas library is used to work with tabular data easily
import pandas as pd

# sqlite3 library is used to connect and run SQL queries on SQLite databases
import sqlite3

# Connect to the database file 'ticketly.db'
conn = sqlite3.connect('ticketly.db')

#  Write an SQL query to select all data from the 'tickets' table
query = "SELECT * FROM tickets"

#  Use pandas to execute the query and convert the result into a DataFrame (table)
df = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

#  Print a heading and display the DataFrame with all tickets
print("--- Ticketly Database Table View ---")
df


--- Ticketly Database Table View ---


,id,movie_name,price,seat_number
0,1,The Legend of Maula Jatt,1000,B-5
1,2,The Legend of Maula Jatt,1000,B-5
2,3,The Legend of Maula Jatt,1000,B-5


#Saving Database Data as Excel or CSV

In [ ]:
# Save the DataFrame (all tickets) into a CSV file
# index=False means we do NOT save the row numbers in the CSV
df.to_csv('ticketly_report.csv', index=False)

# Print a confirmation message
print(" 'ticketly_report.csv' has been created! Check the 'Files' folder (left side) in Colab.")


 'ticketly_report.csv' has been created! Check the 'Files' folder (left side) in Colab.


In [ ]:
import pandas as pd
import sqlite3       # Used to create, connect, and manage SQLite databases

# Connect to the database file
conn = sqlite3.connect('ticketly.db')

# Read all tickets into a DataFrame
df = pd.read_sql_query("SELECT * FROM tickets", conn)

# Close the connection
conn.close()

# Save the DataFrame to CSV (without the extra index column)
df.to_csv('ticketly_report.csv', index=False)

# Show the DataFrame (this is exactly what will be in the CSV)
print("--- Preview of ticketly_report.csv ---")
df


--- Preview of ticketly_report.csv ---


,id,movie_name,price,seat_number
0,1,The Legend of Maula Jatt,1000,B-5
1,2,The Legend of Maula Jatt,1000,B-5
2,3,The Legend of Maula Jatt,1000,B-5
